In [1]:
%%time
import random
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import torchsummary
from datetime import datetime
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.preprocessing import minmax_scale
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import gc
import os, sys
sys.path.insert(0,'/home/mingyu_seong/Works/func/python')
import mg

rs = 100
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(rs)

def minmax_scale_train_based(train_data, *other_data, eps=1e-8):
    data_min = train_data.min().item()
    data_max = train_data.max().item()
    scale = lambda x: (x - data_min) / (data_max - data_min + eps)
    return (scale(train_data),) + tuple(scale(d) for d in other_data)

def standard_scale_train_based(train_data, *other_data, eps=1e-8):
    data_mean = train_data.mean().item()
    data_std = train_data.std().item()
    scale = lambda x: (x - data_mean) / (data_std + eps)
    return (scale(train_data),) + tuple(scale(d) for d in other_data)

#=================================================================================================
# Define Model (NN)
#=================================================================================================
class NNModel(nn.Module):
    def __init__(self):
        super(NNModel, self).__init__()
        self.dense1 = nn.Linear(train_x.shape[1], 16)
        self.dense2 = nn.Linear(16, 8)
        self.output_frp = nn.Linear(8, 1)

        self.dropout = nn.Dropout(0.2)
        self.batch_norm1 = nn.BatchNorm1d(16)
        self.batch_norm2 = nn.BatchNorm1d(8)

    def forward(self, x):
        x = F.relu(self.batch_norm1(self.dense1(x)))
        x = self.dropout(x)
        x = F.relu(self.batch_norm2(self.dense2(x)))
        return self.output_frp(x)
        
#=================================================================================================
# GPU setting
#=================================================================================================
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

CPU times: user 1.51 s, sys: 195 ms, total: 1.7 s
Wall time: 1.73 s


### Observed data for standardization

In [2]:
#--- Period
syr = 2001
eyr = 2024
nyr = eyr - syr + 1
tim = 'mon'

#--- Clim period
clim_syr = 2001
clim_eyr = 2020

#--- Variables to estimate monthly GPP
feature_names_ordered = [
    'T_clim','P_clim','SM_clim',               # Clims
    'T_anom','P_anom','SM_anom','SW_anom',     # Anomaly
    'SW',                                      # Saturation
    'CO2', 'TCO2', 'SMCO2',                    # CO2 fertilization
    'TSQ']                                     # Stress

#--- Directories
idir = '../../../../_nc/input/OBS/background/r' + str(clim_syr)[2:4] + str(clim_eyr)[2:4] + '/'
idir_clim = idir + 'clim/'
idir_anom = idir + 'anom/'
idir_raw  = idir + 'raw/'

#--- Input files (Climatology & Anomaly)
fin_tas = 'tas_' + tim + '_ERA5_V0_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'
fin_prcp  = 'pr_' + tim + '_GPM_L3_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'
fin_sm  = 'sfcsm_' + tim + '_ERA5_V0_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'
fin_rsds = 'rsds_' + tim + '_MODIS_C62_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'
fin_co2 =  'co2_' + tim + '_noaa_gml_KOR_01deg_' + str(syr) + '-' + str(eyr) + '.nc'

data_files = {
    'T_clim': (idir_clim, fin_tas, 'clim'),
    'P_clim': (idir_clim, fin_prcp, 'clim'),
    'SM_clim': (idir_clim, fin_sm, 'clim'),
    'SW_clim': (idir_clim, fin_rsds, 'clim'),
    'CO2_clim': (idir_clim, fin_co2, 'clim'),
    'T_anom': (idir_anom, fin_tas, 'anom'),
    'P_anom': (idir_anom, fin_prcp, 'anom'),
    'SM_anom': (idir_anom, fin_sm, 'anom'),
    'SW_anom': (idir_anom, fin_rsds, 'anom'),
    'CO2_anom': (idir_anom, fin_co2, 'anom'),
}

#--- Read data
raw_data_arrays = {}
for var_name, (path, filename, da_name) in data_files.items():
    full_path = os.path.join(path, filename)
    try:
        with xr.open_dataset(full_path) as ncid:
            da = ncid[da_name].sel(time=slice(str(syr), str(eyr)))
            raw_data_arrays[var_name] = da
            
    except FileNotFoundError:
        print(f"  오류: 파일이 존재하지 않습니다: {full_path}")
        sys.exit(1)

#--- Raw time series
C0 = 1. # np.nanmean(raw_data_arrays['CO2_clim'].data)
ts_tas = raw_data_arrays['T_anom'] + raw_data_arrays['T_clim']
ts_pr  = raw_data_arrays['P_anom'] + raw_data_arrays['P_clim']
ts_sm  = raw_data_arrays['SM_anom'] + raw_data_arrays['SM_clim']
ts_sw  = raw_data_arrays['SW_anom'] + raw_data_arrays['SW_clim']
ts_co2 = raw_data_arrays['CO2_anom'] + raw_data_arrays['CO2_clim']

#--- Saturation terms
raw_data_arrays['T']   =  ts_tas / ( np.nanmean(raw_data_arrays['T_clim'].data)  + ts_tas)
raw_data_arrays['P']   = ts_pr / ( np.nanmean(raw_data_arrays['P_clim'].data)  + ts_pr) # np.log(ts_pr)
raw_data_arrays['SM']  = ts_sm / ( np.nanmean(raw_data_arrays['SM_clim'].data) + ts_sm) # np.log(ts_sm)
raw_data_arrays['SW']  = ts_sw / ( np.nanmean(raw_data_arrays['SW_clim'].data) + ts_sw) # np.log(ts_sw)
raw_data_arrays['CO2'] = np.log(ts_co2/C0)

#--- Interaction terms
raw_data_arrays['TCO2'] = raw_data_arrays['T_anom'] * np.log(ts_co2/C0)
raw_data_arrays['SMCO2'] = raw_data_arrays['SM_anom'] * np.log(ts_co2/C0)
raw_data_arrays['TSM'] = raw_data_arrays['T_anom'] * raw_data_arrays['SM_anom']
raw_data_arrays['TSQ'] = raw_data_arrays['T_anom']**2

#--- input & label
features_ds = xr.Dataset()
for feature_name in feature_names_ordered:
    da = raw_data_arrays[feature_name]
    features_ds[feature_name] = da

#--- Add Month of Year
if tim == 'mon':
    moy = []
    for imon in np.arange(12):
        mon = imon + 1
        moy.append(float(np.sin(2*np.pi/12*mon)))
    features_ds['MOY'] = xr.DataArray(moy*nyr,
                            dims=['time'],
                            coords={'time': xr.open_dataset(idir_clim+fin_tas).time.data})
    feature_names_ordered = feature_names_ordered + ['MOY']

#--- Combined inputs
train_combined_features_da = features_ds.to_array(dim='variable')

데이터셋으로 통합 및 스태킹 중...


### Inference future monthly GPP (HadGEM3-RA)

In [3]:
#--- Period
syr = 2001
eyr = 2100
nyr = eyr - syr + 1

#--- Scenarios
sce = ['rcp26','rcp45','rcp60','rcp85']

#--- Directories
idir_wgt = './wgt/'
idir = '../../../../_nc/input/HadGEM3-RA/background/r' + str(clim_syr)[2:4] + str(clim_eyr)[2:4] + '/'
idir_clim = idir + 'clim/'
idir_anom = idir + 'anom/'

#--- Input files (Climatology)
fin_tas_clim = 'tas_mon_ERA5_V0_KOR_01deg_200101-210012.nc'
fin_pr_clim  = 'pr_mon_GPM_L3_KOR_01deg_200101-210012.nc'
fin_sm_clim  = 'mrso_mon_ERA5_V0_KOR_01deg_200101-210012.nc'
fin_sw_clim  = 'rsds_mon_MODIS_C62_KOR_01deg_200101-210012.nc'
fin_co2_clim = 'co2_mon_noaa_gml_KOR_01deg_200101-210012.nc'

for isce in sce:
    #--- Directory for output
    odir = '../../../../_nc/prediction/HadGEM3-RA/GPP/background/' + isce + '/'
    os.makedirs(odir, exist_ok=True) 
    
    #--- Input files (Anomaly)
    fin_tas_anom = 'tas_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_200101-210012.nc'
    fin_pr_anom  = 'pr_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_200101-210012.nc'
    fin_sm_anom  = 'mrso_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_200101-210012.nc'
    fin_sw_anom  = 'rsds_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_200101-210012.nc'
    fin_co2_anom = 'co2_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_200101-210012.nc'
    
    data_files = {
        'T_clim': (idir_clim, fin_tas_clim, 'clim'),
        'P_clim': (idir_clim, fin_pr_clim, 'clim'),
        'SM_clim': (idir_clim, fin_sm_clim, 'clim'),
        'SW_clim': (idir_clim, fin_sw_clim, 'clim'),
        'CO2_clim': (idir_clim, fin_co2_clim, 'clim'),
        'T_anom': (idir_anom, fin_tas_anom, 'anom'),
        'P_anom': (idir_anom, fin_pr_anom, 'anom'),
        'SM_anom': (idir_anom, fin_sm_anom, 'anom'),
        'SW_anom': (idir_anom, fin_sw_anom, 'anom'),
        'CO2_anom': (idir_anom, fin_co2_anom, 'anom'),
    }
    
    #--- Read data
    raw_data_arrays = {}
    for var_name, (path, filename, da_name) in data_files.items():
        full_path = os.path.join(path, filename)
        try:
            with xr.open_dataset(full_path) as ncid:
                da = ncid[da_name].sel(time=slice(str(syr), str(eyr)))
                raw_data_arrays[var_name] = da
                
        except FileNotFoundError:
            print(f"  오류: 파일이 존재하지 않습니다: {full_path}")
            sys.exit(1)
    
    #--- Raw time series
    C0 = 1. # np.nanmean(raw_data_arrays['CO2_clim'].data)
    ts_tas = raw_data_arrays['T_anom'] + raw_data_arrays['T_clim']
    ts_pr  = raw_data_arrays['P_anom'] + raw_data_arrays['P_clim']
    ts_sm  = raw_data_arrays['SM_anom'] + raw_data_arrays['SM_clim']
    ts_sw  = raw_data_arrays['SW_anom'] + raw_data_arrays['SW_clim']
    ts_co2 = raw_data_arrays['CO2_anom'] + raw_data_arrays['CO2_clim']
    
    #--- Saturation terms
    raw_data_arrays['T']   = ts_tas / ( np.nanmean(raw_data_arrays['T_clim'].data)  + ts_tas)
    raw_data_arrays['P']   = ts_pr / ( np.nanmean(raw_data_arrays['P_clim'].data)  + ts_pr) # np.log(ts_pr)
    raw_data_arrays['SM']  = ts_sm / ( np.nanmean(raw_data_arrays['SM_clim'].data) + ts_sm) # np.log(ts_sm)
    raw_data_arrays['SW']  = ts_sw / ( np.nanmean(raw_data_arrays['SW_clim'].data) + ts_sw) # np.log(ts_sw)
    raw_data_arrays['CO2'] = np.log(ts_co2/C0)
    
    #--- Interaction terms
    raw_data_arrays['TCO2'] = raw_data_arrays['T_anom'] * np.log(ts_co2/C0)
    raw_data_arrays['SMCO2'] = raw_data_arrays['SM_anom'] * np.log(ts_co2/C0)
    raw_data_arrays['TSM'] = raw_data_arrays['T_anom'] * raw_data_arrays['SM_anom']
    raw_data_arrays['TSQ'] = raw_data_arrays['T_anom']**2
    
    feature_names_ordered = [f for f in feature_names_ordered if f != 'MOY']
    features_ds = xr.Dataset()
    for feature_name in feature_names_ordered:
        da = raw_data_arrays[feature_name]
        features_ds[feature_name] = da
    
    #--- Add Month of Year
    if tim == 'mon':
        moy = []
        for imon in np.arange(12):
            mon = imon + 1
            moy.append(float(np.sin(2*np.pi/12*mon)))
        features_ds['MOY'] = xr.DataArray(moy*nyr,
                                dims=['time'],
                                coords={'time': xr.open_dataset(idir_clim+fin_tas_clim).time.data})
        feature_names_ordered = feature_names_ordered + ['MOY']
        
    #--- Combined inputs
    combined_features_da = features_ds.to_array(dim='variable')
    
    for ifold in np.arange(12):
        train_start  = None ; train_end  = None
        train_start0 = None ; train_end0 = None
        train_start1 = None ; train_end1 = None
        valid_start  = None ; valid_end  = None
        valid_start0 = None ; valid_end0 = None
        valid_start1 = None ; valid_end1 = None
        test_start   = None ; test_end   = None
    
        # Test: 17-yr / Validation: 5-yr / Test: 2-yr   (r: 0.90 / Reg. Coef: 1.00)
        if ifold == 0:
            train_start = '2003' ; train_end = '2019'
            valid_start = '2020' ; valid_end = '2024'
            test_start  = '2001' ; test_end  = '2002'
        elif ifold == 1:
            train_start  = '2005' ; train_end  = '2021'
            valid_start0 = '2022' ; valid_end0 = '2024'
            valid_start1 = '2001' ; valid_end1 = '2002'
            test_start   = '2003' ; test_end   = '2004'
        elif ifold == 2:   
            train_start  = '2005' ; train_end  = '2023'
            valid_start0 = '2024' ; valid_end0 = '2024'
            valid_start1 = '2001' ; valid_end1 = '2004'
            test_start   = '2005' ; test_end   = '2006'
        elif ifold == 3:   
            train_start0 = '2001' ; train_end0 = '2001'
            train_start1 = '2009' ; train_end1 = '2024'
            valid_start  = '2002' ; valid_end  = '2006'
            test_start   = '2007' ; test_end   = '2008'
        elif ifold == 4:   
            train_start0 = '2001' ; train_end0 = '2003'
            train_start1 = '2011' ; train_end1 = '2024'
            valid_start  = '2004' ; valid_end  = '2008'
            test_start   = '2009' ; test_end   = '2010'
        elif ifold == 5:   
            train_start0 = '2001' ; train_end0 = '2005'
            train_start1 = '2013' ; train_end1 = '2024'
            valid_start  = '2006' ; valid_end  = '2010'
            test_start   = '2011' ; test_end   = '2012'
        elif ifold == 6:   
            train_start0 = '2001' ; train_end0 = '2007'
            train_start1 = '2015' ; train_end1 = '2024'
            valid_start  = '2008' ; valid_end  = '2012'
            test_start   = '2013' ; test_end   = '2014'
        elif ifold == 7:   
            train_start0 = '2001' ; train_end0 = '2009'
            train_start1 = '2017' ; train_end1 = '2024'
            valid_start  = '2010' ; valid_end  = '2014'
            test_start   = '2015' ; test_end   = '2016'
        elif ifold == 8:   
            train_start0 = '2001' ; train_end0 = '2011'
            train_start1 = '2019' ; train_end1 = '2024'
            valid_start  = '2012' ; valid_end  = '2016'
            test_start   = '2017' ; test_end   = '2018'
        elif ifold == 9:   
            train_start0 = '2001' ; train_end0 = '2013'
            train_start1 = '2021' ; train_end1 = '2024'
            valid_start  = '2014' ; valid_end  = '2018'
            test_start   = '2019' ; test_end   = '2020'
        elif ifold == 10:   
            train_start0 = '2001' ; train_end0 = '2015'
            train_start1 = '2023' ; train_end1 = '2024'
            valid_start  = '2016' ; valid_end  = '2020'
            test_start   = '2021' ; test_end   = '2022'
        elif ifold == 11:   
            train_start = '2001' ; train_end = '2017'
            valid_start = '2018' ; valid_end = '2022'
            test_start  = '2023' ; test_end  = '2024'
    
        #--- Train
        if train_start0 is not None and train_start1 is not None:
            train_features_da_part0 = train_combined_features_da.sel(time=slice(train_start0, train_end0)) # First period
            train_features_da_part1 = train_combined_features_da.sel(time=slice(train_start1, train_end1)) # Second period
            train_features_da = xr.concat([train_features_da_part0, train_features_da_part1], dim='time')  # Merge periods
        else:
            train_features_da = train_combined_features_da.sel(time=slice(train_start, train_end))
    
        #--- Validation
        if valid_start0 is not None and valid_start1 is not None:
            valid_features_da_part0 = train_combined_features_da.sel(time=slice(valid_start0, valid_end0)) # First period
            valid_features_da_part1 = train_combined_features_da.sel(time=slice(valid_start1, valid_end1)) # Second period
            valid_features_da = xr.concat([valid_features_da_part0, valid_features_da_part1], dim='time')  # Merge periods
        else:
            valid_features_da = train_combined_features_da.sel(time=slice(valid_start, valid_end))
    
        #--- Test
        test_features_da = combined_features_da.sel(time=slice(str(syr), str(eyr)))
    
        stacked_train_features_list, stacked_valid_features_list, stacked_test_features_list = [],[],[]
        for i, feature_name in enumerate(feature_names_ordered):
            stacked_train_feature_array = train_features_da[i].values
            stacked_valid_feature_array  = valid_features_da[i].values
            stacked_test_feature_array  = test_features_da[i].values
            stacked_train_features_list.append(stacked_train_feature_array)
            stacked_valid_features_list.append(stacked_valid_feature_array)
            stacked_test_features_list.append(stacked_test_feature_array)
        
        #--- Separate data (train / validation / test)
        train_x = np.transpose(np.vstack(stacked_train_features_list))
        valid_x = np.transpose(np.vstack(stacked_valid_features_list))
        test_x  = np.transpose(np.vstack(stacked_test_features_list))
    
        #--- Remove NaN
        nan_x_rows = np.isnan(train_x).any(axis=1)
        train_x = train_x[~nan_x_rows]
    
        nan_x_rows = np.isnan(test_x).any(axis=1)
        test_x = test_x[~nan_x_rows]
    
        #--- Normalize variables
        for i in np.arange(train_x.shape[1]):
            train_x[:,i], valid_x[:,i], test_x[:,i] = standard_scale_train_based(train_x[:,i], valid_x[:,i], test_x[:,i])
    
        # Convert numpy arrays to PyTorch tensors
        test_x = torch.tensor(test_x, dtype=torch.float32, device=device)
        
        # Dataset and DataLoader for batching
        test_data = TensorDataset(test_x)
        test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
    
        #=================================================================================================
        # Model Testing
        #=================================================================================================
        model = NNModel().to(device)   # Sending parameters to GPU
        fin_wgt = 'GPP_wgt_' + tim + '_fold' + str(ifold).zfill(2) + '_test_period_' + test_start + '-' + test_end + '.pth'
        model.load_state_dict(torch.load(idir_wgt + fin_wgt, weights_only=False))
        model.eval()
    
        #--- Inference
        y_pred = []
        with torch.no_grad():
            for batch_x, in test_loader:
                outputs = model(batch_x)
                y_pred.extend(outputs.squeeze().cpu().numpy())
        pred = np.array(y_pred)
        
        #--- Save
        pred_data = xr.DataArray(pred,
                                dims=['time'],
                                coords={'time': combined_features_da[0].time.data},
                                attrs={'long_name': 'Predicted GPP',
                                       'units': 'gC/m^2/day'}) #, '_FillValue': 1e+30})
        
        fou = 'predicted_GPP_' + tim + '_' + isce + '_' + str(syr) + '01-' + str(eyr) + \
              '12_fold' + str(ifold).zfill(2) + '_seed' + str(rs) + '.nc'
        ds = xr.Dataset()
        ds['gpp'] = pred_data
        ds.to_netcdf(odir + fou)

    #--- Ensemble mean
    fou = 'predicted_GPP_' + tim + '_' + isce + '_' + str(syr) + '01-' + str(eyr) + \
          '12_ensmean_seed' + str(rs) + '.nc'
    ftmp = mg.get_files(pin=odir,fin='predicted_GPP_' + tim + '_' + isce + '_' + str(syr) + \
                                     '01-' + str(eyr) + '12_fold*_seed' + str(rs) + '.nc')
    flist = [os.path.join(odir, filename ) for filename  in ftmp]
    mg.cdo('ensmean',input=flist, output=odir+fou, options='-s -L -O')

sys.exit()

SystemExit: 

/home/mingyu_seong/Works/lib/anaconda3/envs/torch/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3557: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
